<a href="https://colab.research.google.com/github/addagadanamratha/ms-tct-action-detection-service/blob/main/notebooks/MS_TCT_Milestone3_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!mkdir -p /content/space

In [11]:
%%writefile /content/space/requirements.txt
fastapi
uvicorn[standard]
huggingface_hub
numpy
timm
einops
python-multipart

Overwriting /content/space/requirements.txt


In [4]:
%%writefile /content/space/Dockerfile
FROM python:3.11-slim

RUN apt-get update && apt-get install -y --no-install-recommends git && rm -rf /var/lib/apt/lists/*

# CPU-only PyTorch (small and enough for our model)
RUN pip install --no-cache-dir torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu

COPY requirements.txt /tmp/requirements.txt
RUN pip install --no-cache-dir -r /tmp/requirements.txt

# Official MS-TCT architecture code (Dai et al., CVPR 2022)
RUN git clone --depth 1 https://github.com/dairui01/MS-TCT.git /opt/MS-TCT

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user PATH=/home/user/.local/bin:$PATH PYTHONUNBUFFERED=1
WORKDIR /home/user/app
COPY --chown=user . /home/user/app

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]

Writing /content/space/Dockerfile


In [24]:
%%writefile /content/space/app.py
"""MS-TCT Action Detection as a Service — Team 7 (v2: API + web GUI)"""
import io, os, sys, json, time
import numpy as np, torch
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
from huggingface_hub import snapshot_download

sys.path.insert(0, "/opt/MS-TCT")
from MSTCT.MSTCT_Model import MSTCT

# ---------- Load model + assets from the Hugging Face model repo ----------
MODEL_REPO = os.environ.get("MODEL_REPO", "namratha-addagada/ms-tct-charades")
ASSETS = snapshot_download(MODEL_REPO)
ck = torch.load(f"{ASSETS}/mstct_charades.pth", map_location="cpu", weights_only=True)
model = MSTCT([256, 384, 576, 864], 3, 8, 8, 4096, 512, 157)
model.load_state_dict({k: v.float() for k, v in ck["model_state_dict"].items()}, strict=True)
model.eval()
torch.set_num_threads(os.cpu_count() or 2)

CLASSES = json.load(open(f"{ASSETS}/charades_classes.json"))
SAMPLES = json.load(open(f"{ASSETS}/samples/samples.json"))
FPS, WIN = 6.0, 128          # 6 feature snippets per second, 128-snippet windows
MAX_UPLOAD = 50 * 1024 * 1024

# ---------- Inference ----------
def predict_scores(feats: np.ndarray) -> np.ndarray:
    """[T, 4096] features -> [T, 157] probabilities (sliding 128-snippet windows)."""
    T = len(feats); n = -(-T // WIN)
    x = np.zeros((n * WIN, 4096), np.float32); x[:T] = feats
    x = torch.from_numpy(x).view(n, WIN, 4096).permute(0, 2, 1)
    with torch.inference_mode():
        logits, _ = model(x)
    return torch.sigmoid(logits).reshape(n * WIN, -1)[:T].numpy()

def find_segments(curve, step):
    """Read [start, end] seconds off one class's score curve."""
    k = 5; c = np.convolve(np.pad(curve, (k // 2, k // 2), mode="edge"), np.ones(k) / k, "valid")
    on = c >= max(0.05, 0.6 * c.max())
    segs, start = [], None
    for i, a in enumerate(list(on) + [False]):
        if a and start is None: start = i
        elif not a and start is not None:
            s, e = start * step, i * step
            if segs and s - segs[-1][1] <= 1.0: segs[-1][1] = e      # merge gaps < 1 s
            else: segs.append([s, e])
            start = None
    return [{"start": round(s, 1), "end": round(e, 1)} for s, e in segs if e - s >= 0.5]

def summarize(scores, duration, top_k=5, gt_actions=None):
    T = len(scores); step = duration / T
    video_scores = scores.mean(0)
    top = np.argsort(-video_scores)[:top_k]
    gt = {int(a[0]) for a in gt_actions} if gt_actions else set()
    out = {"duration": round(duration, 2), "num_snippets": T, "top_actions": [
        {"class_id": int(c), "name": CLASSES[c], "score": round(float(video_scores[c]), 4),
         "segments": find_segments(scores[:, c], step),
         "in_ground_truth": (int(c) in gt) if gt_actions else None} for c in top]}
    if gt_actions:
        out["ground_truth"] = [{"class_id": int(c), "name": CLASSES[int(c)],
                                "start": float(s), "end": float(e)} for c, s, e in gt_actions]
        out["correct_in_top_k"] = sorted(int(c) for c in set(top.tolist()) & gt)
    return out

# ---------- API ----------
app = FastAPI(title="MS-TCT Action Detection API",
              description="Temporal action detection (Charades, 157 classes)")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/api/health")
def health():
    return {"status": "ok", "model_repo": MODEL_REPO, "trained_epoch": ck.get("epoch"),
            "val_mAP_300_videos": ck.get("eval_frame_mAP_300val"),
            "trained_on": ck.get("trained_on"), "num_classes": len(CLASSES),
            "num_samples": len(SAMPLES)}

@app.get("/api/samples")
def samples():
    return [{"video_id": v, "duration": i["duration"],
             "ground_truth": sorted({CLASSES[int(a[0])] for a in i["actions"]})}
                        for v, i in SAMPLES.items()]

class SampleRequest(BaseModel):
    video_id: str
    top_k: int = 5

@app.post("/api/predict/sample")
def predict_sample(req: SampleRequest):
    if req.video_id not in SAMPLES:
        raise HTTPException(404, f"Unknown video_id. Choose from {sorted(SAMPLES)}")
    t0 = time.time()
    info = SAMPLES[req.video_id]
    feats = np.load(f"{ASSETS}/samples/{req.video_id}.npy").astype(np.float32)
    result = summarize(predict_scores(feats), info["duration"],
                       max(1, min(req.top_k, 20)), info["actions"])
    result.update({"video_id": req.video_id, "latency_ms": round((time.time() - t0) * 1000, 1)})
    return result

@app.post("/api/predict/features")
async def predict_features(file: UploadFile = File(...), top_k: int = Form(5)):
    """Upload your own features: a .npy array of shape [T, 4096] (6 snippets per second)."""
    data = await file.read()
    if len(data) > MAX_UPLOAD:
        raise HTTPException(413, "File too large (max 50 MB).")
    try:
        feats = np.asarray(np.load(io.BytesIO(data), allow_pickle=False), dtype=np.float32)
    except Exception:
        raise HTTPException(400, "Not a valid .npy file.")
    if feats.ndim == 2 and feats.shape[0] == 4096 and feats.shape[1] != 4096:
        feats = feats.T                                  # accept [4096, T] too
    if feats.ndim != 2 or feats.shape[1] != 4096:
        raise HTTPException(400, f"Expected shape [T, 4096], got {list(feats.shape)}.")
    if not 1 <= len(feats) <= 20000:
        raise HTTPException(400, "T must be between 1 and 20000 snippets.")
    t0 = time.time()
    result = summarize(predict_scores(feats), len(feats) / FPS, max(1, min(top_k, 20)))
    result.update({"filename": file.filename, "request_bytes": len(data),
                   "latency_ms": round((time.time() - t0) * 1000, 1)})
    return result

# ---------- Web GUI (served at "/") — must be mounted last ----------
app.mount("/", StaticFiles(directory="web", html=True), name="web")

Overwriting /content/space/app.py


In [29]:
# Upload server v1 to the Space
from google.colab import userdata
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
api.upload_folder(folder_path="/content/space",
                  repo_id="namratha-addagada/ms-tct-action-detection",
                  repo_type="space", commit_message="Server v2: web GUI + feature upload")
print("Uploaded! Watch the build: https://huggingface.co/spaces/namratha-addagada/ms-tct-action-detection")

Uploaded! Watch the build: https://huggingface.co/spaces/namratha-addagada/ms-tct-action-detection


In [28]:
# Put the web GUI into /content/space/web/
import os, shutil
from google.colab import files
os.makedirs("/content/space/web", exist_ok=True)
uploaded = files.upload()                       # choose index.html in the dialog
shutil.move(list(uploaded)[0], "/content/space/web/index.html")
print(os.listdir("/content/space"), os.listdir("/content/space/web"))

Saving index.html to index.html
['web', 'Dockerfile', 'app.py', 'requirements.txt'] ['index.html']


In [16]:
# Get the 12 demo videos from the Charades archive, make them web-ready, upload
!pip -q install remotezip
import json, os, subprocess
from remotezip import RemoteZip
from huggingface_hub import hf_hub_download, HfApi
from google.colab import userdata

REPO = "namratha-addagada/ms-tct-charades"
samples = json.load(open(hf_hub_download(REPO, "samples/samples.json")))
ids = sorted(samples)
RAW, WEB = "/content/videos_raw", "/content/videos_web"
os.makedirs(RAW, exist_ok=True); os.makedirs(WEB, exist_ok=True)

# 1) Pull only our 12 videos out of the 13 GB zip (HTTP range requests, no full download)
urls = ["https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades_v1_480.zip",
        "http://ai2-website.s3.amazonaws.com/data/Charades_v1_480.zip"]
for url in urls:
    try:
        with RemoteZip(url) as z:
            names = {os.path.basename(n)[:-4]: n for n in z.namelist() if n.endswith(".mp4")}
            print("Opened archive:", url, "| videos inside:", len(names))
            for vid in ids:
                z.extract(names[vid], RAW)
                print("  extracted", vid)
        break
    except Exception as e:
        print("Failed with", url, "->", type(e).__name__, e)

# 2) Web-friendly MP4 (360p, H.264, instant start) + thumbnail
for vid in ids:
    src = next(os.path.join(r, f) for r, _, fs in os.walk(RAW) for f in fs if f == f"{vid}.mp4")
    out = f"{WEB}/{vid}.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vf", "scale=-2:360",
                    "-c:v", "libx264", "-preset", "veryfast", "-crf", "28", "-pix_fmt", "yuv420p",
                    "-an", "-movflags", "+faststart", out], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", "1", "-i", out,
                    "-frames:v", "1", "-vf", "scale=320:-2", f"{WEB}/{vid}.jpg"], check=True)
    print(f"{vid}: {os.path.getsize(out)/1e6:.1f} MB  (annotated length {samples[vid]['duration']} s)")

# 3) Upload to the model repo under videos/
api = HfApi(token=userdata.get("HF_TOKEN"))
api.upload_folder(folder_path=WEB, path_in_repo="videos", repo_id=REPO, repo_type="model",
                  commit_message="Add 12 demo videos + thumbnails")
print("\nUploaded. Example:", f"https://huggingface.co/{REPO}/resolve/main/videos/{ids[0]}.mp4")

samples.json:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

Opened archive: https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades_v1_480.zip | videos inside: 9848
  extracted 0Z1PC
  extracted 1F706
  extracted 1FIGA
  extracted 6B2FN
  extracted 6XB10
  extracted 7E2AJ
  extracted AEKYE
  extracted EIT66
  extracted FK143
  extracted HQ8BB
  extracted QRSRE
  extracted WM2J7
0Z1PC: 0.5 MB  (annotated length 38.83 s)
1F706: 0.2 MB  (annotated length 30.04 s)
1FIGA: 0.4 MB  (annotated length 35.75 s)
6B2FN: 0.4 MB  (annotated length 23.92 s)
6XB10: 0.8 MB  (annotated length 35.75 s)
7E2AJ: 0.2 MB  (annotated length 16.25 s)
AEKYE: 0.2 MB  (annotated length 32.58 s)
EIT66: 0.1 MB  (annotated length 12.08 s)
FK143: 0.3 MB  (annotated length 36.46 s)
HQ8BB: 0.3 MB  (annotated length 16.08 s)
QRSRE: 0.3 MB  (annotated length 33.0 s)
WM2J7: 0.5 MB  (annotated length 30.46 s)

Uploaded. Example: https://huggingface.co/namratha-addagada/ms-tct-charades/resolve/main/videos/0Z1PC.mp4


In [21]:
# STEP 19 — Use the model's best validation videos as demo examples
import io, json, os, tarfile, subprocess, shutil
import json, os, tarfile, subprocess, shutil
import pandas as pd, numpy as np
from google.colab import drive, userdata
from huggingface_hub import HfApi
!pip -q install remotezip
from remotezip import RemoteZip
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/CCN-Project"
REPO, SPACE = "namratha-addagada/ms-tct-charades", "namratha-addagada/ms-tct-action-detection"
if not os.path.exists("/content/MS-TCT"):
    !git clone -q https://github.com/dairui01/MS-TCT.git /content/MS-TCT
ann = json.load(open("/content/MS-TCT/data/charades.json"))

# 1) Rank the 300 validation videos by how well the model did (from Step 10)
df = pd.read_csv(f"{PROJECT}/results/large_best_per_video.csv")
df["duration"] = df.video.map(lambda v: ann[v]["duration"])
df = df[(df.duration >= 12) & (df.duration <= 40)]               # demo-friendly lengths
best = df.sort_values(["f1", "correct"], ascending=False).head(10)
typical = df.iloc[(df.f1 - df.f1.median()).abs().argsort()[:2]]  # 2 average cases, for honesty
chosen = pd.concat([best, typical]).drop_duplicates("video")
print(chosen[["video", "correct", "precision", "recall", "f1", "duration"]].to_string(index=False))
ids = list(chosen.video)

# 2) Features from your Drive backup (only these videos)
OUT = "/content/new_demo"; shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(f"{OUT}/samples"); os.makedirs(f"{OUT}/videos")
want = {f"charades_npy_large/{v}.npy" for v in ids}
with tarfile.open(f"{PROJECT}/MS_TCT_Large/charades_npy_large.tar") as tar:
    for m in tar:
        if m.name in want:
            arr = np.load(io.BytesIO(tar.extractfile(m).read()))
            np.save(f"{OUT}/samples/{os.path.basename(m.name)}", arr.astype(np.float16))
print("features extracted:", len(os.listdir(f"{OUT}/samples")))

# samples.json keeps the order above: best first, typical last
json.dump({v: {"duration": ann[v]["duration"], "actions": ann[v]["actions"]} for v in ids},
          open(f"{OUT}/samples/samples.json", "w"), indent=1)

# 3) Videos + thumbnails from the Charades zip
with RemoteZip("https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades_v1_480.zip") as z:
    names = {os.path.basename(n)[:-4]: n for n in z.namelist() if n.endswith(".mp4")}
    for v in ids:
        z.extract(names[v], "/content/raw480")
        src = f"/content/raw480/{names[v]}"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vf", "scale=-2:360",
                        "-c:v", "libx264", "-preset", "veryfast", "-crf", "28", "-pix_fmt", "yuv420p",
                        "-an", "-movflags", "+faststart", f"{OUT}/videos/{v}.mp4"], check=True)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", "1", "-i", f"{OUT}/videos/{v}.mp4",
                        "-frames:v", "1", "-vf", "scale=320:-2", f"{OUT}/videos/{v}.jpg"], check=True)
print("videos ready:", len([f for f in os.listdir(f"{OUT}/videos") if f.endswith(".mp4")]))

# 4) Replace the old demo set on the Hub and restart the Space
api = HfApi(token=userdata.get("HF_TOKEN"))
api.upload_folder(folder_path=OUT, repo_id=REPO, repo_type="model",
                  delete_patterns=["samples/*", "videos/*"],
                  commit_message="Demo set: model's best validation videos + 2 typical cases")
api.restart_space(SPACE)
print("Done. The Space is restarting with the new examples.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
video  correct  precision   recall       f1  duration
B47TS        5        1.0 0.714286 0.833333     21.79
HXR3Z        5        1.0 0.555556 0.714286     33.42
8H570        4        0.8 0.571429 0.666667     30.75
WLV4Q        4        0.8 0.500000 0.615385     29.83
KVXJ9        3        0.6 0.600000 0.600000     30.75
0KZYF        4        0.8 0.444444 0.571429     30.71
0HGNK        5        1.0 0.384615 0.555556     35.21
HQ8BB        3        0.6 0.500000 0.545455     16.08
IFRT0        3        0.6 0.500000 0.545455     30.58
EHIOY        4        0.8 0.400000 0.533333     34.71
TM0BV        1        0.2 0.142857 0.166667     16.17
QRSRE        1        0.2 0.142857 0.166667     33.00
features extracted: 12
videos ready: 12
Done. The Space is restarting with the new examples.


In [30]:
# BACKUP — copy the live Space (server + web app) and the demo set to Drive
from google.colab import drive, userdata
from huggingface_hub import snapshot_download
import shutil
drive.mount("/content/drive")
tok = userdata.get("HF_TOKEN")
B = "/content/drive/MyDrive/CCN-Project/backup_2026-11-19"
snapshot_download("namratha-addagada/ms-tct-action-detection", repo_type="space",
                  local_dir=f"{B}/space_server_code", token=tok)
snapshot_download("namratha-addagada/ms-tct-charades", repo_type="model",
                  local_dir=f"{B}/model_repo", token=tok,
                  allow_patterns=["*.json", "*.md", "samples/*", "videos/*"])
shutil.rmtree(f"{B}/space_server_code/.cache", ignore_errors=True)
shutil.rmtree(f"{B}/model_repo/.cache", ignore_errors=True)
print("Backed up to", B)
!ls -R "{B}" | head -40

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

Backed up to /content/drive/MyDrive/CCN-Project/backup_2026-11-19
/content/drive/MyDrive/CCN-Project/backup_2026-11-19:
model_repo
space_server_code

/content/drive/MyDrive/CCN-Project/backup_2026-11-19/model_repo:
charades_classes.json
feature_stats.json
README.md
samples
videos

/content/drive/MyDrive/CCN-Project/backup_2026-11-19/model_repo/samples:
0HGNK.npy
0KZYF.npy
8H570.npy
B47TS.npy
EHIOY.npy
HQ8BB.npy
HXR3Z.npy
IFRT0.npy
KVXJ9.npy
QRSRE.npy
samples.json
TM0BV.npy
WLV4Q.npy

/content/drive/MyDrive/CCN-Project/backup_2026-11-19/model_repo/videos:
0HGNK.jpg
0HGNK.mp4
0KZYF.jpg
0KZYF.mp4
8H570.jpg
8H570.mp4
B47TS.jpg
B47TS.mp4
EHIOY.jpg
EHIOY.mp4
HQ8BB.jpg
HQ8BB.mp4
HXR3Z.jpg
